In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from DATA.stock_invest_function import *


In [2]:
def calculate_correlation_between_dfs(df1, df2, start_date=None, end_date=None, method='pearson', min_periods=4):
    """
    두 개의 시계열 DataFrame의 상관관계를 계산하되, 유효 관측치가 min_periods보다 많을 경우만 수행

    Parameters:
    ...
    - min_periods (int): 최소 유효 데이터 수

    Returns:
    - pd.DataFrame: 상관계수 매트릭스
    """
    if start_date:
        df1 = df1[df1.index >= pd.to_datetime(start_date)]
        df2 = df2[df2.index >= pd.to_datetime(start_date)]
    if end_date:
        df1 = df1[df1.index <= pd.to_datetime(end_date)]
        df2 = df2[df2.index <= pd.to_datetime(end_date)]

    combined = pd.merge(df1, df2, left_index=True, right_index=True, how='inner', suffixes=('_firm', '_hs'))

    corr_matrix = pd.DataFrame(index=df1.columns, columns=df2.columns, dtype=float)

    for firm in df1.columns:
        for hs in df2.columns:
            x = combined[firm]
            y = combined[hs]
            valid = x.notna() & y.notna()
            if valid.sum() >= min_periods:
                corr_matrix.loc[firm, hs] = x[valid].corr(y[valid], method=method)
            else:
                corr_matrix.loc[firm, hs] = np.nan  # 또는 0

    return corr_matrix

def get_top_correlated_hscode(corr_matrix, symbol, top_n=5, threshold=None, ascending=False):
    """
    특정 기업(Symbol)에 대해 상관관계가 높은 HS 코드를 추출하는 함수

    Parameters:
    - corr_matrix (pd.DataFrame): Symbol x HS_Code 형태의 상관관계 행렬
    - symbol (str): 대상 Symbol (예: '000080')
    - top_n (int): 상위 N개 추출 (threshold와 함께 사용 시 무시될 수 있음)
    - threshold (float or None): 상관계수 하한값 (예: 0.5 이상만 보기). 설정 시 top_n보다 우선함
    - ascending (bool): 상관계수 기준 오름차순 정렬 여부 (기본값: False = 높은 값 우선)

    Returns:
    - pd.DataFrame: root_hs_code 및 상관계수를 포함한 상위 N개 HS 코드
    """

    if symbol not in corr_matrix.index:
        raise ValueError(f"Symbol '{symbol}' not found in correlation matrix.")

    symbol_corr = corr_matrix.loc[symbol].dropna()

    if threshold is not None:
        symbol_corr = symbol_corr[symbol_corr >= threshold]

    top_correlated = symbol_corr.sort_values(ascending=ascending).head(top_n)

    return top_correlated.reset_index().rename(columns={'index': 'root_hs_code', symbol: 'correlation'})

def get_top_correlated_symbols(corr_matrix, hs_code, top_n=5, threshold=None, ascending=False):
    """
    특정 HS 코드에 대해 상관관계가 높은 기업 Symbol을 추출하는 함수

    Parameters:
    - corr_matrix (pd.DataFrame): Symbol x HS_Code 형태의 상관관계 행렬
    - hs_code (str or int): 대상 HS 코드 (예: '151550')
    - top_n (int): 상위 N개 추출
    - threshold (float or None): 상관계수 하한값 (예: 0.5 이상만 보기)
    - ascending (bool): 정렬 방향 (False: 높은 상관 우선)

    Returns:
    - pd.DataFrame: symbol 및 correlation 정보를 담은 상위 N개 결과
    """

    if hs_code not in corr_matrix.columns:
        raise ValueError(f"HS code '{hs_code}' not found in correlation matrix columns.")

    hs_corr = corr_matrix[hs_code].dropna()

    if threshold is not None:
        hs_corr = hs_corr[hs_corr >= threshold]

    top_symbols = hs_corr.sort_values(ascending=ascending).head(top_n)

    return top_symbols.reset_index().rename(columns={'index': 'symbol', hs_code: 'correlation'})


In [3]:
db_info = {
    'host': get_db_host(),
    # 'host': '192.168.0.230',
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# 테이블 이름
table_name = 'target_hs_code'

# 고유한 hs_code 값 추출 쿼리 실행
query = f"SELECT DISTINCT hs_code FROM {table_name}"
unique_hs_codes_df = pd.read_sql(query, con=engine)
hs_codes  = unique_hs_codes_df['hs_code'].unique().tolist()

indicator = 'expDlr'

df_real = fetch_trade_data_multi_hscode(db_info, hs_codes, indicator, 'korea_monthly_trade_data')

# 분기 정보 추가
df_real['quarter'] = df_real['date'].dt.to_period('Q')

# 그룹별로 분기별 합산
df_quarterly = (
    df_real
    .groupby(['root_hs_code', 'quarter'])['value']
    .sum()
    .reset_index()
)

# 👉 분기 월말로 변환 (예: 2007Q1 → 2007-03-31)
df_quarterly['date'] = df_quarterly['quarter'].dt.to_timestamp(how='end')

# 👉 'quarter' 컬럼 제거
df_quarterly.drop(columns=['quarter'], inplace=True)

# 1단계: 문자열로 직접 변환하려면 to_datetime 이후에 바로 strftime
df_quarterly['date'] = pd.to_datetime(df_quarterly['date']).dt.strftime('%Y-%m-%d')

def create_yoy_growth_pivot(df_quarterly, start_date=None, end_date=None):
    """
    전년 동분기 대비 증가율을 pivot 형태로 변환하고 분석기간을 설정할 수 있는 함수

    Parameters:
    - df_quarterly (DataFrame): 'root_hs_code', 'date', 'yoy_growth' 포함된 데이터
    - start_date (str or None): 분석 시작일 (예: '2015-01-01')
    - end_date (str or None): 분석 종료일 (예: '2023-12-31')

    Returns:
    - pivot_df (DataFrame): 행: date, 열: root_hs_code, 값: yoy_growth
    """
    # Pivot
    pivot_df = df_quarterly.pivot(
        index='date',
        columns='root_hs_code',
        values='yoy_growth'
    ).sort_index()

    # inf 값 NaN 처리
    pivot_df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 분석 기간 슬라이싱 (날짜가 문자열이면 datetime으로 변환)
    pivot_df.index = pd.to_datetime(pivot_df.index)

    if start_date:
        pivot_df = pivot_df[pivot_df.index >= pd.to_datetime(start_date)]
    if end_date:
        pivot_df = pivot_df[pivot_df.index <= pd.to_datetime(end_date)]

    return pivot_df


# 전년 동분기 값 (4개 분기 전 값) 계산
df_quarterly['yoy_value'] = (
    df_quarterly
    .sort_values(['root_hs_code', 'date'])
    .groupby('root_hs_code')['value']
    .shift(4)
)

# ❗ yoy_growth 계산
df_quarterly['yoy_growth'] = (
    (df_quarterly['value'] - df_quarterly['yoy_value']) / df_quarterly['yoy_value']
) * 100

quarterly_trade_data = create_yoy_growth_pivot(df_quarterly, start_date='2008-03', end_date='2025-03')

In [4]:
fs_df = fetch_table_data(db_info, "korea_fs_data")
fs_df.rename(columns={'Date': 'date'}, inplace=True)

# 1. indicator 필터링
target_indicator = '매출액(천원)'
filtered_df = fs_df[fs_df['indicator'] == target_indicator].copy()

# 2. 날짜 정제 및 정렬
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
filtered_df.sort_values(by='date', inplace=True)

# 3. value 컬럼이 있는지 확인 및 타입 강제
if 'value' not in filtered_df.columns:
    raise KeyError("'value' 컬럼이 없습니다.")

filtered_df['value'] = pd.to_numeric(filtered_df['value'], errors='coerce')

# 4. 피벗 테이블 생성 (행: date, 열: Symbol, 값: value)
pivot_df = filtered_df.pivot_table(
    index='date',
    columns='symbol',
    values='value',
    aggfunc='first'  # 중복 방지
)

# 5. 전년 동분기 대비 변화율 계산 (4분기 전 대비)
fs_yoy_growth_df = pivot_df.pct_change(periods=4) * 100

✅ 'korea_fs_data' 테이블에서 5584577건의 데이터를 가져왔습니다.


In [5]:
# correlation_result = calculate_correlation_between_dfs(
#     fs_yoy_growth_df,
#     quarterly_trade_data,
#     start_date='2020-03-31',
#     end_date='2025-03-31'
# )
#
# # 상위 몇 개 확인
# correlation_result.head()

In [7]:
 # correlation_result.to_csv(r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\한국상장사_수출데이터_상관계수_202508.csv", index=True, encoding="utf-8-sig")

path = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\한국상장사_수출데이터_상관계수_202508.csv"
# path = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\한국상장사_수출데이터_상관계수_202508.csv"
correlation_result = pd.read_csv(path).set_index('symbol')

In [13]:
top_symbols = get_top_correlated_symbols(
    corr_matrix=correlation_result,
    hs_code= '853529',
    top_n= 50,
    threshold=0.1  # 선택사항
)
print(top_symbols)

     symbol  correlation
0   A019180     0.842408
1   A011320     0.797856
2   A212560     0.792069
3   A131400     0.771270
4   A053700     0.759457
5   A215100     0.749233
6   A375500     0.741871
7   A015750     0.733331
8   A053270     0.730896
9   A013520     0.727399
10  A101390     0.723963
11  A005810     0.722592
12  A010770     0.722281
13  A033530     0.715382
14  A011210     0.709235
15  A043370     0.708403
16  A001620     0.705604
17  A092780     0.703545
18  A025540     0.703062
19  A018880     0.701987
20  A017510     0.686893
21  A000270     0.677616
22  A013310     0.674261
23  A090080     0.668655
24  A024910     0.666918
25  A261200     0.664719
26  A204320     0.661501
27  A010690     0.661341
28  A051600     0.658625
29  A070960     0.657032
30  A046120     0.656520
31  A025530     0.656192
32  A065510     0.655570
33  A123700     0.653848
34  A050320     0.650658
35  A009680     0.650005
36  A005710     0.649465
37  A122690     0.645160
38  A304360     0.644087


In [8]:
top_hs_codes = get_top_correlated_hscode(
    corr_matrix=correlation_result,  # 이전에 만든 상관관계 행렬
    symbol ='A0',
    top_n=100,
    threshold=0.3  # 선택사항
)

print(top_hs_codes.head(50))

ValueError: Symbol 'A0' not found in correlation matrix.

In [9]:
from pykrx import stock

# 1. ticker 리스트 불러오기
tickers = stock.get_market_ticker_list(market="ALL")

# 2. ticker와 name을 리스트로 만들기
data = []
for t in tickers:
    name = stock.get_market_ticker_name(t)
    data.append({
        'ticker': t,
        'name': name,
        'symbol': 'A' + t
    })

# 3. DataFrame으로 변환
company_name_df = pd.DataFrame(data, columns=['symbol', 'ticker', 'name'])


In [27]:
correlation_result.loc['A068270'][['300214']]

300214    0.462663
Name: A068270, dtype: float64

In [18]:
pd.merge(top_symbols, company_name_df, on='symbol', how = 'left')

,symbol,correlation,ticker,name
0,A004080,0.828144,004080,신흥
1,A016590,0.799607,016590,신대양제지
2,A317770,0.774407,317770,엑스페릭스
3,A238120,0.750952,238120,얼라인드
4,A066670,0.750186,066670,디티씨
5,A115530,0.749475,115530,씨엔플러스
6,A001360,0.740978,001360,삼성제약
7,A003650,0.738403,003650,미창석유
8,A065660,0.733880,065660,안트로젠
9,A005090,0.726738,005090,SGC에너지


In [35]:
hscode = fetch_table_data(db_info, "target_hs_code")

hscode[hscode['hs_code'] == '854232']

✅ 'target_hs_code' 테이블에서 766건의 데이터를 가져왔습니다.


,hs_code
429,854232
